# 使用工具强制输出 JSON

## 学习目标

* 理解如何使用工具来强制结构化响应
* 运用这一"技巧"生成结构化 JSON

利用工具使用的一个更有趣的方式是强制 Claude 以 JSON 这样的结构化内容进行响应。在很多情况下，我们可能需要从 Claude 获取标准化的 JSON 响应：提取实体、总结数据、分析情感等等。

一种方法是直接要求 Claude 以 JSON 格式响应，但这可能需要额外的工作来从 Claude 返回的大字符串中提取 JSON，或者确保 JSON 遵循我们想要的精确格式。

好消息是，**每当 Claude 想要使用工具时，它已经按照我们在定义工具时告诉它的完美结构化格式进行响应了。**

在上一节课中，我们给 Claude 提供了一个计算器工具。当它想使用这个工具时，它的响应内容是这样的：

```
{
    'operand1': 1984135, 
    'operand2': 9343116, 
    'operation': 'multiply'
}
```

这看起来和 JSON 非常相似！

如果我们想让 Claude 生成结构化 JSON，可以利用这一点。我们只需要定义一个描述特定 JSON 结构的工具，然后告诉 Claude 就行了。就这样。Claude 会返回响应，以为它正在"调用工具"，但实际上我们关心的只是它给我们的结构化响应。

***

# 概念概述

这与我们在上一节课中所做的有什么不同？以下是上一节课工作流程的图表：



在上一节课中，我们给 Claude 提供了访问某个工具的权限，Claude 想要调用它，然后我们实际调用了底层的工具函数。

在这节课中，我们将"欺骗"Claude，告诉它有关某个工具的信息，但我们不需要实际调用底层的工具函数。我们使用工具作为强制特定响应结构的方式，如下图所示：



## 情感分析
让我们从一个简单的例子开始。假设我们想让 Claude 分析某段文本的情感，并按照以下格式返回一个 JSON 对象：

```
{
  "negative_score": 0.6,
  "neutral_score": 0.3,
  "positive_score": 0.1
}
```

我们只需要定义一个使用 JSON Schema 描述这个格式的工具。以下是一个可能的实现：

In [15]:
tools = [
    {
        "name": "print_sentiment_scores",
        "description": "Prints the sentiment scores of a given text.",
        "input_schema": {
            "type": "object",
            "properties": {
                "positive_score": {"type": "number", "description": "The positive sentiment score, ranging from 0.0 to 1.0."},
                "negative_score": {"type": "number", "description": "The negative sentiment score, ranging from 0.0 to 1.0."},
                "neutral_score": {"type": "number", "description": "The neutral sentiment score, ranging from 0.0 to 1.0."}
            },
            "required": ["positive_score", "negative_score", "neutral_score"]
        }
    }
]

现在我们可以告诉 Claude 这个工具，并明确告诉 Claude 使用它，以确保它确实使用了它。我们应该会收到一个响应，告诉我们 Claude 想要使用工具。工具使用响应应该包含我们所需格式的所有数据。

In [5]:
from anthropic import Anthropic
from dotenv import load_dotenv
import json

load_dotenv()
client = Anthropic()

tweet = "I'm a HUGE hater of pickles.  I actually despise pickles.  They are garbage."

query = f"""
<text>
{tweet}
</text>

Only use the print_sentiment_scores tool.
"""

response = client.messages.create(
    model="claude-3-sonnet-20240229",
    max_tokens=4096,
    tools=tools,
    messages=[{"role": "user", "content": query}]
)

In [6]:
response

ToolsBetaMessage(id='msg_01BhF4TkK8vDM6z5m4FNGRnB', content=[TextBlock(text='Here is the sentiment analysis for the given text:', type='text'), ToolUseBlock(id='toolu_01Mt1an3KHEz5RduZRUUuTWz', input={'positive_score': 0.0, 'negative_score': 0.791, 'neutral_score': 0.209}, name='print_sentiment_scores', type='tool_use')], model='claude-3-sonnet-20240229', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(input_tokens=374, output_tokens=112))

让我们看看从 Claude 那里得到的响应。我们已经将重要部分加粗了：


>ToolsBetaMessage(id='msg_01BhF4TkK8vDM6z5m4FNGRnB', content=[TextBlock(text='Here is the sentiment analysis for the given text:', >type='text'), ToolUseBlock(id='toolu_01Mt1an3KHEz5RduZRUUuTWz', **input={'positive_score': 0.0, 'negative_score': 0.791, 'neutral_score': 0.209}**, name='print_sentiment_scores', type='tool_use')], model='claude-3-sonnet-20240229', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(input_tokens=374, output_tokens=112)
)

Claude"以为"它正在调用一个将使用此情感分析数据的工具，但实际上我们只是提取数据并将其转换为 JSON：

In [9]:
import json
json_sentiment = None
for content in response.content:
    if content.type == "tool_use" and content.name == "print_sentiment_scores":
        json_sentiment = content.input
        break

if json_sentiment:
    print("Sentiment Analysis (JSON):")
    print(json.dumps(json_sentiment, indent=2))
else:
    print("No sentiment analysis found in the response.")

Sentiment Analysis (JSON):
{
  "positive_score": 0.0,
  "negative_score": 0.791,
  "neutral_score": 0.209
}


成功了！现在让我们把它转换成一个可重用的函数，接收一条推文或文章，然后以 JSON 格式打印或返回情感分析结果。

In [ ]:
def analyze_sentiment(content):

    query = f"""
    <text>
    {content}
    </text>

    Only use the print_sentiment_scores tool.
    """

    response = client.messages.create(
        model="claude-3-sonnet-20240229",
        max_tokens=4096,
        tools=tools,
        messages=[{"role": "user", "content": query}]
    )

    json_sentiment = None
    for content in response.content:
        if content.type == "tool_use" and content.name == "print_sentiment_scores":
            json_sentiment = content.input
            break

    if json_sentiment:
        print("Sentiment Analysis (JSON):")
        print(json.dumps(json_sentiment, indent=2))
    else:
        print("No sentiment analysis found in the response.")


In [11]:
analyze_sentiment("OMG I absolutely love taking bubble baths soooo much!!!!")

Sentiment Analysis (JSON):
{
  "positive_score": 0.8,
  "negative_score": 0.0,
  "neutral_score": 0.2
}


In [12]:
analyze_sentiment("Honestly I have no opinion on taking baths")

Sentiment Analysis (JSON):
{
  "positive_score": 0.056,
  "negative_score": 0.065,
  "neutral_score": 0.879
}


***

## 使用 `tool_choice` 强制工具使用

目前我们通过提示来"强制"Claude 使用我们的 `print_sentiment_scores` 工具。在我们的提示中，我们写`Only use the print_sentiment_scores tool.`，这通常有效，但有一个更好的方法！我们实际上可以使用 `tool_choice` 参数来强制 Claude 使用特定工具：

In [ ]:
tool_choice={"type": "tool", "name": "print_sentiment_scores"}

上面的代码告诉 Claude，它必须通过调用 `print_sentiment_scores` 工具来响应。让我们更新我们的函数来使用它：

In [ ]:
def analyze_sentiment(content):

    query = f"""
    <text>
    {content}
    </text>

    Only use the print_sentiment_scores tool.
    """

    response = client.messages.create(
        model="claude-3-sonnet-20240229",
        max_tokens=4096,
        tools=tools,
        tool_choice={"type": "tool", "name": "print_sentiment_scores"},
        messages=[{"role": "user", "content": query}]
    )

    json_sentiment = None
    for content in response.content:
        if content.type == "tool_use" and content.name == "print_sentiment_scores":
            json_sentiment = content.input
            break

    if json_sentiment:
        print("Sentiment Analysis (JSON):")
        print(json.dumps(json_sentiment, indent=2))
    else:
        print("No sentiment analysis found in the response.")

我们将在后续课程中更详细地介绍 `tool_choice`。

***

## 实体提取示例

让我们使用同样的方法让 Claude 生成格式良好的 JSON，其中包含从文本样本中提取的人名、组织名和地点等实体：

In [14]:
tools = [
    {
        "name": "print_entities",
        "description": "Prints extract named entities.",
        "input_schema": {
            "type": "object",
            "properties": {
                "entities": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string", "description": "The extracted entity name."},
                            "type": {"type": "string", "description": "The entity type (e.g., PERSON, ORGANIZATION, LOCATION)."},
                            "context": {"type": "string", "description": "The context in which the entity appears in the text."}
                        },
                        "required": ["name", "type", "context"]
                    }
                }
            },
            "required": ["entities"]
        }
    }
]

text = "John works at Google in New York. He met with Sarah, the CEO of Acme Inc., last week in San Francisco."

query = f"""
<document>
{text}
</document>

Use the print_entities tool.
"""

response = client.messages.create(
    model="claude-3-sonnet-20240229",
    max_tokens=4096,
    tools=tools,
    messages=[{"role": "user", "content": query}]
)

json_entities = None
for content in response.content:
    if content.type == "tool_use" and content.name == "print_entities":
        json_entities = content.input
        break

if json_entities:
    print("Extracted Entities (JSON):")
    print(json.dumps(json_entities, indent=2))
else:
    print("No entities found in the response.")

Extracted Entities (JSON):
{
  "entities": [
    {
      "name": "John",
      "type": "PERSON",
      "context": "John works at Google in New York."
    },
    {
      "name": "Google",
      "type": "ORGANIZATION",
      "context": "John works at Google in New York."
    },
    {
      "name": "New York",
      "type": "LOCATION",
      "context": "John works at Google in New York."
    },
    {
      "name": "Sarah",
      "type": "PERSON",
      "context": "He met with Sarah, the CEO of Acme Inc., last week in San Francisco."
    },
    {
      "name": "Acme Inc.",
      "type": "ORGANIZATION",
      "context": "He met with Sarah, the CEO of Acme Inc., last week in San Francisco."
    },
    {
      "name": "San Francisco",
      "type": "LOCATION",
      "context": "He met with Sarah, the CEO of Acme Inc., last week in San Francisco."
    }
  ]
}


我们使用的是和之前一样的"技巧"。我们告诉 Claude 它可以访问某个工具，作为让 Claude 以特定数据格式响应的一种方式。然后我们提取 Claude 响应的格式化数据，就可以使用了。

请记住，在这种用例中，明确告诉 Claude 我们希望它使用某个工具会有帮助：


>Use the print_entities tool.

***

## Wikipedia 摘要示例（更复杂的数据）

让我们尝试另一个稍微复杂的例子。我们将使用 Python 的 `wikipedia` 包获取整个 Wikipedia 文章并将其传递给 Claude。我们将使用 Claude 生成包含以下内容的响应：

* 文章的主题
* 文章的摘要
* 文章中提到的关键词和主题列表
* 文章的分类列表（娱乐、政治、商业等）以及分类分数（即该主题属于该类别的强度）

如果我们把关于 Walt Disney 的 Wikipedia 文章传递给 Claude，我们可能期望得到这样的结果：

```
{
  "subject": "Walt Disney",
  "summary": "Walter Elias Disney was an American animator, film producer, and entrepreneur. He was a pioneer of the American animation industry and introduced several developments in the production of cartoons. He held the record for most Academy Awards earned and nominations by an individual. He was also involved in the development of Disneyland and other theme parks, as well as television programs.",
  "keywords": [
    "Walt Disney",
    "animation",
    "film producer",
    "entrepreneur",
    "Disneyland",
    "theme parks",
    "television"
  ],
  "categories": [
    {
      "name": "Entertainment",
      "score": 0.9
    },
    {
      "name": "Business",
      "score": 0.7
    },
    {
      "name": "Technology",
      "score": 0.6
    }
  ]
}
```

以下是一个函数的可能实现，该函数接收一个 Wikipedia 页面主题，找到文章，下载内容，传递给 Claude，然后打印出结果 JSON 数据。我们使用相同的策略，定义一个工具来"引导"Claude 响应的形状。

注意：如果你的机器上没有安装 `wikipedia`，请确保运行 `pip install wikipedia`！

In [27]:
import wikipedia

#tool definition
tools = [
    {
        "name": "print_article_classification",
        "description": "Prints the classification results.",
        "input_schema": {
            "type": "object",
            "properties": {
                "subject": {
                    "type": "string",
                    "description": "The overall subject of the article",
                },
                "summary": {
                    "type": "string",
                    "description": "A paragaph summary of the article"
                },
                "keywords": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "description": "List of keywords and topics in the article"
                    }
                },
                "categories": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string", "description": "The category name."},
                            "score": {"type": "number", "description": "The classification score for the category, ranging from 0.0 to 1.0."}
                        },
                        "required": ["name", "score"]
                    }
                }
            },
            "required": ["subject","summary", "keywords", "categories"]
        }
    }
]

#The function that generates the json for a given article subject
def generate_json_for_article(subject):
    page = wikipedia.page(subject, auto_suggest=True)
    query = f"""
    <document>
    {page.content}
    </document>

    Use the print_article_classification tool. Example categories are Politics, Sports, Technology, Entertainment, Business.
    """

    response = client.messages.create(
        model="claude-3-haiku-20240307",
        max_tokens=4096,
        tools=tools,
        messages=[{"role": "user", "content": query}]
    )

    json_classification = None
    for content in response.content:
        if content.type == "tool_use" and content.name == "print_article_classification":
            json_classification = content.input
            break

    if json_classification:
        print("Text Classification (JSON):")
        print(json.dumps(json_classification, indent=2))
    else:
        print("No text classification found in the response.")

In [29]:
generate_json_for_article("Jeff Goldblum")

Text Classification (JSON):
{
  "subject": "Jeff Goldblum",
  "summary": "Jeffrey Lynn Goldblum is an American actor and musician who has starred in some of the highest-grossing films, such as Jurassic Park and Independence Day. He has had a long and successful career in both film and television, with roles in a wide range of movies and TV shows. Goldblum is also an accomplished jazz musician and has released several albums with his band, The Mildred Snitzer Orchestra.",
  "keywords": [
    "actor",
    "musician",
    "Jurassic Park",
    "Independence Day",
    "film",
    "television",
    "jazz"
  ],
  "categories": [
    {
      "name": "Entertainment",
      "score": 0.9
    }
  ]
}


In [37]:
generate_json_for_article("Octopus")

Text Classification (JSON):
{
  "subject": "Octopus",
  "summary": "This article provides a comprehensive overview of octopuses, including their anatomy, physiology, behavior, ecology, and evolutionary history. It covers topics such as their complex nervous systems, camouflage and color-changing abilities, intelligence, and relationships with humans.",
  "keywords": [
    "octopus",
    "cephalopod",
    "mollusc",
    "marine biology",
    "animal behavior",
    "evolution"
  ],
  "categories": [
    {
      "name": "Science",
      "score": 0.9
    },
    {
      "name": "Nature",
      "score": 0.8
    }
  ]
}


In [38]:
generate_json_for_article("Herbert Hoover")

Text Classification (JSON):
{
  "subject": "Herbert Hoover",
  "summary": "The article provides a comprehensive biography of Herbert Hoover, the 31st President of the United States. It covers his early life, career as a mining engineer and humanitarian, his presidency during the Great Depression, and his post-presidency activities.",
  "keywords": [
    "Herbert Hoover",
    "Great Depression",
    "Republican Party",
    "U.S. President",
    "mining engineer",
    "Commission for Relief in Belgium",
    "U.S. Food Administration",
    "Secretary of Commerce",
    "Smoot\u2013Hawley Tariff Act",
    "New Deal"
  ],
  "categories": [
    {
      "name": "Politics",
      "score": 0.9
    },
    {
      "name": "Business",
      "score": 0.7
    },
    {
      "name": "History",
      "score": 0.8
    }
  ]
}


***

## 练习

使用上述策略编写一个名为 `translate` 的函数，该函数接收一个单词或短语，并生成结构化的 JSON 输出，包括英语原文以及西班牙语、法语、日语和阿拉伯语的翻译。

以下是一个示例，说明它应该如何工作：

如果我们调用这个函数：

In [ ]:
translate("how much does this cost")

我们期望得到这样的输出：

```json
{
  "english": "how much does this cost",
  "spanish": "¿cuánto cuesta esto?",
  "french": "combien ça coûte?",
  "japanese": "これはいくらですか",
  "arabic": "كم تكلفة هذا؟"
}
```

**注意：如果你想打印结果，这行代码可以帮助你很好地打印它们：**

In [ ]:
print(json.dumps(translations_from_claude, ensure_ascii=False, indent=2))